In [ ]:
# Zachary Katz
# zachary_katz@mines.edu
# 04 June 2025'

import numpy as np
import scipy

In [ ]:
# tide is the tide data
# dates_timeseries is the corresponding datetime 

# Calculate modfied tidal form factor
# 3 days of tides with a 0.5 day slide

# Convert dates_timeseries to seconds since dates_timeseries[0]
seconds = [(date - dates_timeseries[0]).total_seconds() for date in dates_timeseries]

spacing = 4  # Minutes
mean_days = 3
slide_days = 1
mean_units = int(mean_days * 24 * 60 / spacing)
slide_units = int(slide_days * 24 * 60 / spacing)

HR_TO_SEC = 3600
T_O1 = 25.81933871 * HR_TO_SEC
T_K1 = 23.93447213 * HR_TO_SEC
T_M2 = 12.4206012 * HR_TO_SEC
T_S2 = 12 * HR_TO_SEC


def sines(x, A1, phi1, A2, phi2):
    return A1 * np.sin(2 * np.pi * x / ((T_O1 + T_K1) / 2) + phi1) + A2 * np.sin(
        2 * np.pi * x / ((T_M2 + T_S2) / 2) + phi2
    )


form_factors = []
dates_form_factor = []
semidiurnal = []
diurnal = []
# Extract 3 days of tidal data with a 0.5 day slide
start = 0
end = mean_units
while end < len(seconds):
    seconds_tide = np.array(seconds[start:end], dtype=float)
    tide = np.array(tide[start:end], dtype=float)
    date_midpoint = dates_timeseries[(start + end) // 2]
    start += slide_units
    end += slide_units

    # Fit a sum of sines to the tide
    initial_guess = [50, 0, 50, 0]
    popt, pcov = scipy.optimize.curve_fit(sines, seconds_tide, tide, p0=initial_guess)

    # Extract fitted parameters
    Diurnal_fit, phi1_fit, SemiDiurnal_fit, phi2_fit = popt

    # Generate the fitted curve
    y_fit = sines(seconds_tide, Diurnal_fit, phi1_fit, SemiDiurnal_fit, phi2_fit)
    form_factor = np.abs(Diurnal_fit / SemiDiurnal_fit)
    semidiurnal.append((SemiDiurnal_fit))
    diurnal.append((Diurnal_fit))

    form_factors.append(form_factor)
    dates_form_factor.append(date_midpoint)

    # fig, ax = plt.subplots()
    # ax.plot(dates_form_factor, form_factors, color="black")